<a href="https://colab.research.google.com/github/ThiagoLira19/classificador_de_categorias/blob/main/Classificador_de_categorias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch

In [2]:
from transformers import pipeline

In [3]:
# Carrega o pipeline de Zero-Shot Classification com um modelo multilíngue (XLM-RoBERTa)
# O modelo será baixado automaticamente na primeira execução (~2.2 GB)
classificador = pipeline(
    "zero-shot-classification", model="joeddav/xlm-roberta-large-xnli"
)

config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [12]:
# Categorias desejadas
categorias = ["divergencia de valores", "erro de calculo", "endosso"]

# Template em português para ajudar o modelo a entender o contexto
template = "O assunto deste texto é {}."

In [10]:
def categorizar_texto(texto, limite_confianca=0.50):
    resultado = classificador(
        texto,
        candidate_labels=categorias,
        hypothesis_template=template,
        multi_label=False,  # Define False para escolher apenas a categoria principal
    )

    # Retorna a categoria com maior pontuação e o nível de confiança
    melhor_categoria = resultado["labels"][0]
    confianca = resultado["scores"][0]

    # Regra de fallback: se a confiança for menor que o limite, classifica como 'outros'
    if confianca < limite_confianca:
        return "outros", confianca

    return melhor_categoria, confianca

In [14]:
# --- Exemplos de uso ---
textos_teste = [
    "O valor cobrado na fatura é de R$ 500,00, mas o contrato acordado era de R$ 450,00.",
    "A soma dos itens da nota fiscal está incorreta, 100 + 50 deveria dar 150 e não 180.",
    "Solicito a inclusão de um novo veículo na apólice vigente a partir de hoje.",
    "Gostaria de saber qual é o horário de atendimento da filial.",
]

# Você pode ajustar o limite_confianca (ex: 0.45, 0.50, 0.60) conforme seus testes
LIMITE = 0.50

for texto in textos_teste:
    categoria, score = categorizar_texto(texto, limite_confianca=LIMITE)
    print(f"Texto: '{texto}'")
    print(f"-> Categoria: {categoria.upper()} (Confiança: {score:.2%})\n")

Texto: 'O valor cobrado na fatura é de R$ 500,00, mas o contrato acordado era de R$ 450,00.'
-> Categoria: DIVERGENCIA DE VALORES (Confiança: 90.62%)

Texto: 'A soma dos itens da nota fiscal está incorreta, 100 + 50 deveria dar 150 e não 180.'
-> Categoria: ERRO DE CALCULO (Confiança: 76.11%)

Texto: 'Solicito a inclusão de um novo veículo na apólice vigente a partir de hoje.'
-> Categoria: ENDOSSO (Confiança: 78.40%)

Texto: 'Gostaria de saber qual é o horário de atendimento da filial.'
-> Categoria: OUTROS (Confiança: 42.44%)

